# CPM AI-Assisted Interpretation
## AI Diagnosis Layer for Control Performance Monitoring

---

### Purpose

This notebook implements an **AI-assisted interpretation layer** on top of the
deterministic CPM analysis produced by `CPM_Use_Case_GIMSCOP.ipynb`.

It does **not** recalculate any KPIs or access raw process data.
It reads the pre-computed CSV outputs, builds a structured payload per loop,
and calls a language model to translate KPI patterns into plain-language
engineering interpretations and prioritised action recommendations.

---

### Architecture

```
CPM_Use_Case_GIMSCOP.ipynb
    │
    ├── outputs/cpm_kpi_windows_flagged.csv   ← 77 window-level KPI rows + flags
    ├── outputs/cpm_loop_summary.csv          ← 11 loop-level aggregated summary
    └── outputs/cpm_kpi_windows.csv           ← raw windowed KPIs (for cascade check)
         │
         ▼
CPM_AI_Interpretation.ipynb  (this notebook)
    │
    ├── Builds a compact structured payload per loop
    │       KPI summary + diagnosis flags + positioner metrics + data quality notes
    │
    ├── Calls Mistral AI API (via OpenAI-compatible endpoint)
    │       The model receives KPI context, not raw data
    │       Returns structured JSON: priority / labels / interpretation /
    │                                evidence / recommended_checks / confidence
    │
    └── outputs/cpm_ai_diagnosis.csv          ← AI diagnosis table, one row per loop
```

---

### Design principles

1. **Deterministic first**: all KPI calculations are done in the CPM notebook using
   transparent, reproducible Python code. The AI layer is only an interpretation step.

2. **Structured input**: the model receives a compact JSON payload with named fields,
   not free-form text or raw time series. This reduces hallucination risk and keeps
   token cost low (~500 input tokens per loop).

3. **Cautious language**: the system prompt explicitly instructs the model to use
   screening language ("consistent with", "possible") rather than confirmed diagnoses.
   Root-cause confirmation always requires additional plant context.

4. **Auditable output**: every diagnosis in the exported CSV can be traced back to
   the KPI values in the payload. Nothing is hidden.

---

### Prerequisites

- `CPM_Use_Case_GIMSCOP.ipynb` has been executed and its outputs are in `outputs/`
- A Mistral API key is stored in a `.env` file at the project root (never in this notebook)
- `pip install openai python-dotenv` has been run in this environment

---

### Limitations

The AI interpretation inherits the limitations of the CPM analysis:
- No process topology — multi-loop interaction cannot be confirmed
- No controller MODE signal — manual-mode periods cannot be excluded
- Physical valve limits not available — positioner diagnostics are screening indicators
- SP rescaling applied to FIC29 and FIC38 — error-based KPIs are approximate for these loops


## 1. Environment Setup

Load the API key from the `.env` file. The key is never stored in this notebook.

In [11]:
from dotenv import load_dotenv
import os

load_dotenv()   # reads .env from the project root

api_key = os.getenv("MISTRAL_API_KEY")
if not api_key:
    raise EnvironmentError(
        "MISTRAL_API_KEY not found. "
        "Create a .env file in the project root with: MISTRAL_API_KEY=your_key"
    )

print("API key loaded ✓")


API key loaded ✓


## 2. Load CPM Analysis Outputs

The three CSV files produced by `CPM_Use_Case_GIMSCOP.ipynb` are loaded:

- **cpm_kpi_windows_flagged.csv**: 77 rows (11 loops × 7 windows), KPIs + diagnosis flags
- **cpm_loop_summary.csv**: 11 rows, aggregated KPIs per loop (mean, max across windows)
- **cpm_kpi_windows.csv**: 77 rows, full KPI table used for cascade slave detection


In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

OUTPUTS = Path("outputs")

kpi_windows   = pd.read_csv(OUTPUTS / "cpm_kpi_windows_flagged.csv")
loop_summary  = pd.read_csv(OUTPUTS / "cpm_loop_summary.csv")
kpi_raw       = pd.read_csv(OUTPUTS / "cpm_kpi_windows.csv")

TAGS = sorted(kpi_windows["loop_tag"].unique())

print(f"Loops loaded       : {len(TAGS)}")
print(f"Windows per loop   : {len(kpi_windows) // len(TAGS)}")
print(f"Total observations : {len(kpi_windows)}")
print(f"Loop tags          : {TAGS}")
print()
print("Summary columns:", list(loop_summary.columns))


Loops loaded       : 11
Windows per loop   : 7
Total observations : 77
Loop tags          : ['FIC11', 'FIC12', 'FIC24', 'FIC28', 'FIC29', 'FIC30', 'FIC38', 'LIC42', 'LIC43', 'PIC47', 'TIC52']

Summary columns: ['loop_tag', 'mean_oi_zc', 'max_oi_zc', 'mean_oi_reg', 'max_oi_reg', 'max_oi_reg_raw', 'mean_p_osc', 'mean_iae_norm', 'max_abs_bias', 'bias_std', 'mean_sp_std', 'mean_tv_op_norm', 'mean_op_mv_mae', 'mean_tv_mv_ratio']


## 3. API Client

Mistral AI is accessed via the OpenAI-compatible endpoint.
This avoids the `mistralai` SDK compatibility issue with Python 3.13.

**Model used:** `mistral-small-latest` — sufficient quality for structured JSON
diagnosis at low token cost. Replace with `mistral-large-latest` for higher
reasoning depth if needed.


In [13]:
from openai import OpenAI

client = OpenAI(
    api_key  = api_key,
    base_url = "https://api.mistral.ai/v1"
)

MODEL = "mistral-small-latest"
print(f"Client ready — model: {MODEL}")


Client ready — model: mistral-small-latest


## 4. Payload Builder

The payload is a compact structured dict that summarises one loop for the model.

**Design decisions:**
- Only aggregated KPI statistics are sent (mean, max), not raw time series
- The full `bias_trend` list (7 values) is included to allow the model to detect
  temporal degradation patterns (e.g., monotonically growing bias in LIC42)
- `spectral_power_fraction_pct` is explicitly named to prevent the model from
  confusing it with an oscillation period
- `oi_reg_max_raw` is capped at 500 before sending to prevent numerical overflow
  artifacts in the model response (values above 500 carry the same diagnostic
  meaning: highly regular oscillation)
- `cascade_slave` flag is derived from SP behaviour: if SP has many unique values
  and varies significantly across windows, the loop is likely driven by a master
  controller
- `all_windows_flagged` helps the model distinguish persistent from intermittent issues


In [14]:
import json

def build_payload(tag: str) -> dict:
    rows     = kpi_windows[kpi_windows["loop_tag"] == tag].sort_values("window")
    summary  = loop_summary[loop_summary["loop_tag"] == tag].iloc[0]
    raw_rows = kpi_raw[kpi_raw["loop_tag"] == tag].sort_values("window")

    # ── Diagnosis flags ───────────────────────────────────────────────────────
    all_flags = []
    for f in rows["diagnosis_flags"]:
        all_flags.extend([x.strip() for x in str(f).split(",") if x.strip()])
    flag_counts = pd.Series(all_flags).value_counts().to_dict()

    # ── OI_reg overflow cap ───────────────────────────────────────────────────
    oi_raw = summary.get("max_oi_reg_raw", np.nan)
    oi_reg_max_raw = (
        None if (pd.isna(oi_raw) or float(oi_raw) > 500)
        else round(float(oi_raw), 2)
    )

    # ── Pre-computed threshold classifications ────────────────────────────────
    tv_ratio = float(summary["mean_tv_mv_ratio"])
    mv_valid = bool(not pd.isna(tv_ratio) and tv_ratio < 3.0)

    if not mv_valid:
        tv_category, tv_below_0_6, tv_below_0_8 = "mv_not_valve_position", False, False
    elif tv_ratio < 0.60:
        tv_category, tv_below_0_6, tv_below_0_8 = "attenuated_response", True, True
    elif tv_ratio < 0.80:
        tv_category, tv_below_0_6, tv_below_0_8 = "slightly_attenuated", False, True
    else:
        tv_category, tv_below_0_6, tv_below_0_8 = "normal", False, False

    oi_zc_mean = float(summary["mean_oi_zc"])
    oi_zc_category = (
        "high_activity" if oi_zc_mean > 0.30 else
        "moderate"      if oi_zc_mean > 0.10 else
        "low"
    )

    oi_reg_mean = float(summary["mean_oi_reg"]) if not pd.isna(summary["mean_oi_reg"]) else None
    oi_reg_category = (
        None           if oi_reg_mean is None else
        "very_regular" if oi_reg_mean > 5.0  else
        "regular"      if oi_reg_mean > 1.0  else
        "irregular"
    )

    iae_mean = float(summary["mean_iae_norm"])
    iae_category = (
        "poor_tracking" if iae_mean > 2.0 else
        "moderate"      if iae_mean > 0.5 else
        "good"
    )

    bias_max = float(summary["max_abs_bias"])
    bias_category = (
        "significant" if bias_max > 0.10 else
        "minor"       if bias_max > 0.05 else
        "negligible"
    )

    op_mv_mae = float(summary["mean_op_mv_mae"])
    op_mv_category = (
        "large_offset" if op_mv_mae > 0.10 else
        "moderate"     if op_mv_mae > 0.05 else
        "small"
    )

    # ── Cascade slave detection — uses sp_std from loop_summary ──────────────
    # sp_std = std of SP values within each window (computed in CPM notebook)
    # High sp_std → SP is being driven continuously by a master controller
    # Low sp_std  → SP is constant (set by operator) or stepped occasionally
    mean_sp_std = float(summary.get("mean_sp_std", 0.0))
    cascade_slave = bool(mean_sp_std > 0.05 and not bool(rows["sp_rescaled"].iloc[0]))
    # FIC28: sp_std high (master varies SP continuously)  → True  ✓
    # FIC11: sp_std ≈ 0 (SP constant, own oscillation)   → False ✓
    # LIC42: sp_std ≈ 0 (SP = −0.095 unchanged)          → False ✓

    # ── Persistence ───────────────────────────────────────────────────────────
    n_flagged = int((rows["diagnosis_flags"] != "normal_or_low_priority").sum())
    n_windows = int(len(rows))
    persistence = (
        "persistent"   if n_flagged == n_windows else
        "frequent"     if n_flagged >= n_windows * 0.5 else
        "intermittent" if n_flagged > 0 else
        "none"
    )

    # ── Bias trend direction ──────────────────────────────────────────────────
    bias_trend = [round(float(r["bias"]), 3) for _, r in rows.iterrows()]
    bias_direction = (
        "growing"  if bias_trend[-1] - bias_trend[0] >  0.05 else
        "shrinking" if bias_trend[-1] - bias_trend[0] < -0.05 else
        "stable"
    )

    return {
        "loop_tag":        tag,
        "instrument_type": tag[:3],
        "n_windows":       n_windows,

        "oscillation": {
            "oi_zc_mean":                  round(oi_zc_mean, 3),
            "oi_zc_max":                   round(float(summary["max_oi_zc"]), 3),
            "oi_zc_category":              oi_zc_category,
            "oi_reg_mean":                 round(oi_reg_mean, 2) if oi_reg_mean else None,
            "oi_reg_max_raw":              oi_reg_max_raw,
            "oi_reg_category":             oi_reg_category,
            "oi_reg_gt1_windows":          int((rows["oi_reg"].fillna(0) > 1.0).sum()),
            "spectral_power_fraction_pct": round(float(summary["mean_p_osc"]), 1),
        },

        "tracking": {
            "iae_norm_mean":   round(iae_mean, 3),
            "iae_category":    iae_category,
            "bias_max_abs":    round(bias_max, 3),
            "bias_category":   bias_category,
            "bias_trend":      bias_trend,
            "bias_direction":  bias_direction,
        },

        "valve_activity": {
            "tv_op_norm_mean": round(float(summary["mean_tv_op_norm"]), 4),
        },

        "positioner": {
            "op_mv_mae":              round(op_mv_mae, 3),
            "op_mv_offset_category":  op_mv_category,
            "tv_mv_op_ratio":         round(tv_ratio, 3),
            "tv_mv_op_below_0_60":    tv_below_0_6,
            "tv_mv_op_below_0_80":    tv_below_0_8,
            "tv_mv_op_category":      tv_category,
            "mv_is_valve_position":   mv_valid,
        },

        "context": {
            "cascade_slave":       cascade_slave,
            "persistence":         persistence,
            "all_windows_flagged": bool(n_flagged == n_windows),
            "windows_flagged":     n_flagged,
            "bias_direction":      bias_direction,
            "sp_rescaled":         bool(rows["sp_rescaled"].iloc[0]),
        },

        "flags":    flag_counts,

        "known_limitations": [
            "No process topology — multi-loop interaction cannot be confirmed",
            "No controller MODE signal — manual mode periods cannot be excluded",
            "Physical valve limits not available — positioner metrics are screening only",
        ]
    }


# Quick validation: build payload for one loop and print it
test_payload = build_payload("FIC28")
print(json.dumps(test_payload, indent=2))


{
  "loop_tag": "FIC28",
  "instrument_type": "FIC",
  "n_windows": 7,
  "oscillation": {
    "oi_zc_mean": 0.011,
    "oi_zc_max": 0.023,
    "oi_zc_category": "low",
    "oi_reg_mean": 4.03,
    "oi_reg_max_raw": 10.72,
    "oi_reg_category": "regular",
    "oi_reg_gt1_windows": 6,
    "spectral_power_fraction_pct": 59.3
  },
  "tracking": {
    "iae_norm_mean": 0.094,
    "iae_category": "good",
    "bias_max_abs": 0.0,
    "bias_category": "negligible",
    "bias_trend": [
      -0.0,
      0.0,
      -0.0,
      0.0,
      -0.0,
      0.0,
      -0.0
    ],
    "bias_direction": "stable"
  },
  "valve_activity": {
    "tv_op_norm_mean": 0.0054
  },
  "positioner": {
    "op_mv_mae": 0.019,
    "op_mv_offset_category": "small",
    "tv_mv_op_ratio": 0.557,
    "tv_mv_op_below_0_60": true,
    "tv_mv_op_below_0_80": true,
    "tv_mv_op_category": "attenuated_response",
    "mv_is_valve_position": true
  },
  "context": {
    "cascade_slave": true,
    "persistence": "persistent",
  

## 5. System Prompt and Diagnosis Function

The system prompt defines the model's role and output format.

**Key instructions:**
- Interpret, do not re-calculate. The KPI values are the ground truth.
- Use cautious language. The model is a screening tool, not a root-cause engine.
- Do not confuse KPI fields — explicit field descriptions are provided in the payload.
- Output must be valid JSON only, with no preamble or explanation outside the structure.


In [15]:
SYSTEM_PROMPT = """You are a control engineering expert providing AI-assisted
interpretation of Control Performance Monitoring (CPM) results.

You receive a structured JSON payload with pre-computed KPI values AND
pre-computed threshold classifications. Use the classification fields
(fields ending in _category, _classification, or boolean flags) as the
authoritative interpretation. The numeric values are provided for context only.

CONSISTENCY RULE — before writing any threshold comparison in your response:
  1. Check the numeric value
  2. Check the boolean flag (e.g. tv_mv_op_below_0_60)
  3. Check the category label (e.g. tv_mv_op_category)
  All three must agree. If they do not agree, trust the category label.
  Never write a threshold statement that contradicts the boolean flags.

CONFIDENCE CALIBRATION:
- "high": KPI signals are strong and consistent across multiple windows,
  multiple independent indicators agree, no data quality issues (sp_rescaled=false,
  no NaN in key fields). Use when persistence="persistent" AND 2+ flag types agree.
- "medium": mixed signals, only 1-2 windows flagged, OR data quality caveats apply
  (sp_rescaled=true, NaN in oi_reg). Default for most cases.
- "low": insufficient evidence, only 1 window flagged, or key KPIs are NaN.

Do NOT default to "medium" for every loop. If persistence="persistent" and
oi_reg_category is "very_regular" or "regular", confidence should be "high".


FIELD MEANINGS:
- oi_zc: zero-crossing rate. Low for slow oscillations even when real.
  Use oi_zc_category, not the raw number, for threshold statements.
- oi_reg_category: "very_regular" (>5), "regular" (1-5), "irregular" (<1)
- spectral_power_fraction_pct: % of signal power at dominant frequency.
  This is NOT a time period. Do not describe it as "oscillation period".
- tv_mv_op_category: pre-computed valve response classification.
  Use this — do not recompute from tv_mv_op_ratio.
- cascade_slave: true means SP is driven by a master controller upstream.
  The oscillation source may be upstream, not in this loop.
- persistence: "persistent" = flagged in all windows; "frequent" = >50% of windows.

LANGUAGE: Use cautious screening language throughout.
  Write "consistent with" or "possible" — never "confirmed" or "detected".
  Final diagnoses require plant verification.

Return ONLY valid JSON with exactly this structure:
{
  "priority": "high|medium|low",
  "labels": ["2-4 short diagnostic labels"],
  "interpretation": "2-3 sentence engineering interpretation",
  "main_evidence": ["3-5 key observations using category labels, not raw thresholds"],
  "recommended_checks": ["3-5 specific actionable next steps"],
  "confidence": "high|medium|low",
  "caveats": "1-2 sentences on key limitations"
}"""


def diagnose(tag: str, verbose: bool = False) -> dict:
    """
    Build payload for one loop, call the Mistral API, and return parsed diagnosis.

    Parameters
    ----------
    tag     : loop identifier (e.g. 'FIC28')
    verbose : if True, print the payload sent to the model

    Returns
    -------
    dict with keys: priority, labels, interpretation, main_evidence,
                    recommended_checks, confidence, caveats
    """
    payload = build_payload(tag)

    if verbose:
        print(f"Payload for {tag}:")
        print(json.dumps(payload, indent=2))

    response = client.chat.completions.create(
        model   = MODEL,
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",
             "content": f"Diagnose this control loop:\n{json.dumps(payload, indent=2)}"}
        ],
        response_format = {"type": "json_object"},
        temperature     = 0.1,   # low temperature for consistent, reproducible output
    )

    raw = response.choices[0].message.content.strip()
    return json.loads(raw)


## 6. Run AI Diagnosis — All Loops

Each loop is diagnosed independently. The API is called once per loop
(~500 input tokens, ~200 output tokens). Total cost for 11 loops is negligible
on the Mistral free tier.

To inspect the payload sent for a specific loop, call `diagnose(tag, verbose=True)`.


In [16]:
diagnoses = {}

for tag in TAGS:
    print(f"  {tag}...", end=" ", flush=True)
    try:
        diagnoses[tag] = diagnose(tag)
        p = diagnoses[tag]["priority"]
        c = diagnoses[tag]["confidence"]
        print(f"→ [{p.upper()}]  confidence={c}")
    except Exception as e:
        print(f"ERROR: {e}")

print(f"\nCompleted: {len(diagnoses)}/{len(TAGS)} loops diagnosed")


  FIC11... → [HIGH]  confidence=high
  FIC12... → [MEDIUM]  confidence=medium
  FIC24... → [MEDIUM]  confidence=medium
  FIC28... → [MEDIUM]  confidence=medium
  FIC29... → [MEDIUM]  confidence=medium
  FIC30... → [MEDIUM]  confidence=medium
  FIC38... → [MEDIUM]  confidence=medium
  LIC42... → [HIGH]  confidence=high
  LIC43... → [LOW]  confidence=medium
  PIC47... → [MEDIUM]  confidence=medium
  TIC52... → [MEDIUM]  confidence=medium

Completed: 11/11 loops diagnosed


## 7. Results — Prioritised Diagnosis Summary

Loops are sorted by priority (high → medium → low).


In [17]:
PRIORITY_ORDER = ["high", "medium", "low"]
PRIORITY_ICONS = {"high": "🔴", "medium": "🟡", "low": "🟢"}

for tag, d in sorted(diagnoses.items(),
                     key=lambda x: PRIORITY_ORDER.index(x[1]["priority"])):
    icon = PRIORITY_ICONS.get(d["priority"], "⚪")
    print(f"\n{'═'*65}")
    print(f"  {icon}  {tag}  [{d['priority'].upper()}]  "
          f"confidence={d['confidence']}")
    print(f"  Labels  : {', '.join(d['labels'])}")
    print(f"  {d['interpretation']}")
    print()
    for ev in d["main_evidence"]:
        print(f"    • {ev}")
    print()
    print(f"  Checks  : {'; '.join(d['recommended_checks'])}")
    print(f"  Caveats : {d['caveats']}")



═════════════════════════════════════════════════════════════════
  🔴  FIC11  [HIGH]  confidence=high
  Labels  : persistent_oscillation, very_regular, cascade_slave, slightly_attenuated_valve
  The loop exhibits persistent and very regular oscillations consistent with a control valve issue, despite being in cascade mode. The valve response is slightly attenuated, which may contribute to the observed behavior.

    • Oscillation is very regular (oi_reg_category: very_regular) and persistent across all windows (7/7).
    • Valve response is classified as slightly attenuated (tv_mv_op_category: slightly_attenuated) despite being in cascade mode.
    • Tracking performance remains good (iae_category: good) with negligible bias (bias_category: negligible).

  Checks  : Verify if the master controller in the cascade is introducing oscillations upstream.; Inspect the control valve for stiction, hysteresis, or improper tuning of the positioner.; Check for valve saturation or mechanical issue

### Priority overview

In [18]:
summary_rows = []
for tag, d in sorted(diagnoses.items(),
                     key=lambda x: PRIORITY_ORDER.index(x[1]["priority"])):
    summary_rows.append({
        "loop":        tag,
        "priority":    d["priority"],
        "confidence":  d["confidence"],
        "labels":      ", ".join(d["labels"]),
    })

summary_table = pd.DataFrame(summary_rows)
display(summary_table.style.apply(
    lambda col: [
        "background-color: #f4cccc" if v == "high"
        else "background-color: #fff2cc" if v == "medium"
        else "background-color: #d9ead3"
        for v in col
    ], subset=["priority"]
))


,loop,priority,confidence,labels
0,FIC11,high,high,"persistent_oscillation, very_regular, cascade_slave, slightly_attenuated_valve"
1,LIC42,high,high,"persistent oscillation, growing bias, valve not position"
2,FIC12,medium,medium,"oscillation, positioner offset, cascade slave"
3,FIC24,medium,medium,"oscillation, cascade_slave, positioner_ok"
4,FIC28,medium,medium,"oscillation, attenuated valve response, cascade slave"
5,FIC29,medium,medium,"tracking error, valve normal, intermittent oscillation"
6,FIC30,medium,medium,"oscillation, cascade_slave, valve_not_position"
7,FIC38,medium,medium,"valve response attenuation, tracking bias, regular oscillation"
8,PIC47,medium,medium,"oscillation, positioner offset, tracking"
9,TIC52,medium,medium,"oscillation, regular, tracking moderate, valve normal"


## 8. Export Results

The diagnosis table is exported as `cpm_ai_diagnosis.csv`.

**Columns:**
- `priority`: high / medium / low — overall engineering priority
- `confidence`: model's confidence in the diagnosis given available data
- `labels`: short diagnostic category tags
- `interpretation`: plain-language engineering summary
- `main_evidence`: KPI observations supporting the diagnosis
- `recommended_checks`: specific next steps for engineering/maintenance review
- `caveats`: key limitations that may affect interpretation

This file can be used directly for reporting, linked to maintenance workflows,
or fed into a downstream AI-assisted reporting layer.


In [19]:
export_rows = []
for tag, d in diagnoses.items():
    export_rows.append({
        "loop_tag":           tag,
        "priority":           d["priority"],
        "confidence":         d["confidence"],
        "labels":             "; ".join(d["labels"]),
        "interpretation":     d["interpretation"],
        "main_evidence":      "; ".join(d["main_evidence"]),
        "recommended_checks": "; ".join(d["recommended_checks"]),
        "caveats":            d["caveats"],
    })

out_df = (pd.DataFrame(export_rows)
          .sort_values("priority",
                       key=lambda x: x.map({"high": 0, "medium": 1, "low": 2}))
          .reset_index(drop=True))

out_path = OUTPUTS / "cpm_ai_diagnosis.csv"
out_df.to_csv(out_path, index=False)

print(f"Saved → {out_path}")
print(f"  {len(out_df)} loops  |  "
      f"high={( out_df['priority']=='high').sum()}  "
      f"medium={(out_df['priority']=='medium').sum()}  "
      f"low={(out_df['priority']=='low').sum()}")
display(out_df)


Saved → outputs\cpm_ai_diagnosis.csv
  11 loops  |  high=2  medium=8  low=1


,loop_tag,priority,confidence,labels,interpretation,main_evidence,recommended_checks,caveats
0,FIC11,high,high,persistent_oscillation; very_regular; cascade_...,The loop exhibits persistent and very regular ...,Oscillation is very regular (oi_reg_category: ...,Verify if the master controller in the cascade...,"Cascade mode complicates root-cause analysis, ..."
1,LIC42,high,high,persistent oscillation; growing bias; valve no...,"The loop shows consistent, very regular oscill...",Oscillation is very regular (oi_reg_category='...,Inspect valve positioner calibration and respo...,Physical valve limits and process topology are...
2,FIC24,medium,medium,oscillation; cascade_slave; positioner_ok,The loop shows consistent moderate oscillation...,Oscillation is classified as 'moderate' with '...,Verify if the oscillation persists when the ca...,Cascade configuration complicates root cause a...
3,FIC12,medium,medium,oscillation; positioner offset; cascade slave,The loop shows persistent regular oscillations...,Oscillation indicators are consistent with 're...,Verify the positioner calibration and zero/spa...,The absence of process topology and controller...
4,FIC28,medium,medium,oscillation; attenuated valve response; cascad...,The loop shows consistent regular oscillations...,Oscillation regularity classified as 'regular'...,Verify upstream controller tuning and oscillat...,Cascade configuration limits root-cause attrib...
5,FIC29,medium,medium,tracking error; valve normal; intermittent osc...,The loop exhibits moderate tracking error with...,Tracking error is classified as moderate (iae_...,Verify controller tuning for setpoint tracking...,Positioner metrics are screening only due to l...
6,FIC30,medium,medium,oscillation; cascade_slave; valve_not_position,The loop shows consistent regular oscillations...,Oscillation is very regular (oi_reg_category: ...,Verify if the oscillation persists when the ca...,The cascade slave configuration complicates ro...
7,FIC38,medium,medium,valve response attenuation; tracking bias; reg...,The loop shows consistent valve response atten...,Valve response is classified as 'attenuated_re...,Inspect valve stem travel and positioner calib...,Physical valve limits and process topology are...
8,PIC47,medium,medium,oscillation; positioner offset; tracking,The loop shows persistent regular oscillations...,Oscillation indicators are persistently flagge...,Inspect positioner calibration and zero/span s...,The diagnosis relies on positioner metrics whi...
9,TIC52,medium,medium,oscillation; regular; tracking moderate; valve...,The loop shows consistent regular oscillations...,Oscillation is classified as 'regular' with a ...,Verify controller tuning for potential oscilla...,The oscillation is not persistent across all w...


## Appendix — Debug Single Loop

Use this cell to inspect the payload and response for any specific loop.
Useful for verifying what the model received and whether the diagnosis is reasonable.


In [20]:
# Change this tag to inspect any loop
DEBUG_TAG = "FIC28"

print(f"=== Payload sent for {DEBUG_TAG} ===")
print(json.dumps(build_payload(DEBUG_TAG), indent=2))

print(f"\n=== AI diagnosis for {DEBUG_TAG} ===")
result = diagnose(DEBUG_TAG, verbose=False)
print(json.dumps(result, indent=2))


=== Payload sent for FIC28 ===
{
  "loop_tag": "FIC28",
  "instrument_type": "FIC",
  "n_windows": 7,
  "oscillation": {
    "oi_zc_mean": 0.011,
    "oi_zc_max": 0.023,
    "oi_zc_category": "low",
    "oi_reg_mean": 4.03,
    "oi_reg_max_raw": 10.72,
    "oi_reg_category": "regular",
    "oi_reg_gt1_windows": 6,
    "spectral_power_fraction_pct": 59.3
  },
  "tracking": {
    "iae_norm_mean": 0.094,
    "iae_category": "good",
    "bias_max_abs": 0.0,
    "bias_category": "negligible",
    "bias_trend": [
      -0.0,
      0.0,
      -0.0,
      0.0,
      -0.0,
      0.0,
      -0.0
    ],
    "bias_direction": "stable"
  },
  "valve_activity": {
    "tv_op_norm_mean": 0.0054
  },
  "positioner": {
    "op_mv_mae": 0.019,
    "op_mv_offset_category": "small",
    "tv_mv_op_ratio": 0.557,
    "tv_mv_op_below_0_60": true,
    "tv_mv_op_below_0_80": true,
    "tv_mv_op_category": "attenuated_response",
    "mv_is_valve_position": true
  },
  "context": {
    "cascade_slave": true,
    